Imports

In [1]:
import os
import time
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ee

from pynhd import NLDI
from supporting_scripts import getData, dataprocessing, SNOTEL_Analyzer, mapping

/uufs/chpc.utah.edu/common/home/u0972368/conda-envs/hyriver/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Setup

In [2]:
nldi = NLDI()

usgs_gage_id = "09330000"   # Fremont River near Bicknell, UT
WY = 2019

StartDate = f"{WY-1}-10-01"
EndDate = f"{WY}-09-30"

OutputFolder = "files/SNOTEL"
os.makedirs(OutputFolder, exist_ok=True)
os.makedirs("Figures", exist_ok=True)

EE Login

In [3]:
ee.Authenticate()
ee.Initialize()

Streamflow data

In [4]:
# Get basin polygon from USGS gage
basin = NLDI().get_basins(usgs_gage_id)
geometry = basin.to_crs("EPSG:4326").geometry[0]
basin_polygon_coords = list(geometry.exterior.coords)

daily_NLDAS_df = getData.get_NLDAS_daily(
    basin_polygon_coords,
    begin_date=StartDate,
    end_date=EndDate
)

daily_NLDAS_df.head()

/uufs/chpc.utah.edu/common/home/u0972368/conda-envs/hyriver/lib/python3.10/site-packages/geopandas/geoseries.py:772: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  val = getattr(super(), mtd)(*args, **kwargs)


Authenticating with Earth Engine...
Initializing Earth Engine...
Earth Engine initialized successfully.


,convective_fraction,longwave_radiation,potential_energy,potential_evaporation,pressure,shortwave_radiation,specific_humidity,temperature,total_precipitation,wind_u,wind_v
Date,,,,,,,,,,,
2018-10-01,0.107919,265.928666,12.275103,0.212778,73997.583419,196.268434,0.005725,11.922340,0.089365,0.913642,4.007560
2018-10-02,0.029557,304.154152,13.961216,0.104625,73957.941048,89.003282,0.007564,9.832566,0.144091,-1.751890,3.449391
2018-10-03,0.230969,277.871537,51.612060,0.122942,73882.076787,167.323142,0.007527,9.150452,0.112957,-0.844505,2.330025
2018-10-04,0.316795,279.622465,67.608203,0.177075,73805.375300,187.067744,0.006030,8.990522,0.236266,1.065523,3.564316
2018-10-05,0.006660,260.681259,25.443709,0.141444,73801.376099,176.036476,0.005280,5.840326,0.005817,2.237381,0.429708


SNOTEL download helper

In [5]:
def download_snotel_csv(SiteName, SiteID, StateAbb, StartDate, EndDate, OutputFolder, tries=5):
    out_csv = f"./{OutputFolder}/df_{SiteID}_{StateAbb}_SNTL.csv"

    if os.path.exists(out_csv):
        df = pd.read_csv(out_csv)
        df["Date"] = pd.to_datetime(df["Date"])
        return df.set_index("Date")

    url = (
        "https://wcc.sc.egov.usda.gov/reportGenerator/view_csv/"
        f"customSingleStationReport/daily/start_of_period/{SiteID}:{StateAbb}:SNTL"
        f"%7Cid=%22%22%7Cname/{StartDate},{EndDate}/WTEQ::value"
        "?fitToScreen=false"
    )

    for attempt in range(tries):
        try:
            r = requests.get(url, timeout=60)
            r.raise_for_status()

            with open(out_csv, "w", encoding="utf-8") as f:
                f.write(r.text)

            df = pd.read_csv(out_csv)
            df["Date"] = pd.to_datetime(df["Date"])
            return df.set_index("Date")

        except Exception as e:
            if attempt == tries - 1:
                raise
            wait = 2 ** attempt
            print(f"SNOTEL download failed: {e}")
            print(f"Retrying in {wait} seconds...")
            time.sleep(wait)

Snowtel Data

In [6]:
def download_snotel_csv(SiteName, SiteID, StateAbb, StartDate, EndDate, OutputFolder, tries=5):
    os.makedirs(OutputFolder, exist_ok=True)
    out_csv = f"./{OutputFolder}/df_{SiteID}_{StateAbb}_SNTL.csv"

    if os.path.exists(out_csv):
        df = pd.read_csv(out_csv)
        df["Date"] = pd.to_datetime(df["Date"])
        return df.set_index("Date")

    url = (
        "https://wcc.sc.egov.usda.gov/reportGenerator/view_csv/"
        f"customSingleStationReport/daily/start_of_period/{SiteID}:{StateAbb}:SNTL"
        f"%7Cid=%22%22%7Cname/{StartDate},{EndDate}/WTEQ::value"
        "?fitToScreen=false"
    )

    for attempt in range(tries):
        try:
            r = requests.get(url, timeout=300)
            r.raise_for_status()

            with open(out_csv, "w", encoding="utf-8") as f:
                f.write(r.text)

            df = pd.read_csv(out_csv)
            df["Date"] = pd.to_datetime(df["Date"])
            return df.set_index("Date")

        except Exception as e:
            if attempt == tries - 1:
                raise
            wait = 2 ** attempt
            print(f"SNOTEL download failed: {e}")
            print(f"Retrying in {wait} seconds...")
            time.sleep(wait)

SiteName = "FREMONT RIVER NEAR BICKNELL"
SiteID = "09330000"
StateAbb = "UT"

snotel_df = download_snotel_csv(
    SiteName=SiteName,
    SiteID=SiteID,
    StateAbb=StateAbb,
    StartDate=StartDate,
    EndDate=EndDate,
    OutputFolder=OutputFolder
)

snotel_df.head()

SNOTEL download failed: HTTPSConnectionPool(host='wcc.sc.egov.usda.gov', port=443): Read timed out. (read timeout=300)
Retrying in 1 seconds...
SNOTEL download failed: HTTPSConnectionPool(host='wcc.sc.egov.usda.gov', port=443): Read timed out. (read timeout=300)
Retrying in 2 seconds...
SNOTEL download failed: HTTPSConnectionPool(host='wcc.sc.egov.usda.gov', port=443): Read timed out. (read timeout=300)
Retrying in 4 seconds...
SNOTEL download failed: HTTPSConnectionPool(host='wcc.sc.egov.usda.gov', port=443): Read timed out. (read timeout=300)
Retrying in 8 seconds...


ReadTimeout: HTTPSConnectionPool(host='wcc.sc.egov.usda.gov', port=443): Read timed out. (read timeout=300)

Landsat helper

In [ ]:
def get_landsat_ndsi_timeseries(basin_polygon_coords, begin_date, end_date):
    basin = ee.Geometry.Polygon(basin_polygon_coords)

    col = (
        ee.ImageCollection("LANDSAT/LC08/C02/T2_L2")
        .filterBounds(basin)
        .filterDate(begin_date, end_date)
        .sort("CLOUD_COVER")
    )

    n = col.size().getInfo()
    img_list = col.toList(n)
    records = []

    for i in range(n):
        img = ee.Image(img_list.get(i))

        qa = img.select("QA_PIXEL")
        cloud_mask = (
            qa.bitwiseAnd(1 << 1).eq(0)
            .And(qa.bitwiseAnd(1 << 2).eq(0))
            .And(qa.bitwiseAnd(1 << 3).eq(0))
            .And(qa.bitwiseAnd(1 << 4).eq(0))
        )

        green = img.select("SR_B3").multiply(2.75e-05).add(-0.2)
        swir1 = img.select("SR_B6").multiply(2.75e-05).add(-0.2)

        ndsi = green.subtract(swir1).divide(green.add(swir1)).rename("NDSI")
        ndsi = ndsi.updateMask(cloud_mask)

        stat = ndsi.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=basin,
            scale=30,
            maxPixels=1e9,
        ).getInfo()

        if stat and stat.get("NDSI") is not None:
            records.append({
                "Date": pd.to_datetime(img.date().format("YYYY-MM-dd").getInfo()),
                "NDSI": stat["NDSI"],
            })

    df = pd.DataFrame(records)
    if not df.empty:
        df = df.sort_values("Date").set_index("Date")
    return df

Landsat

In [ ]:
landsat_df = get_landsat_ndsi_timeseries(
    basin_polygon_coords=basin_polygon_coords,
    begin_date=StartDate,
    end_date=EndDate
)

landsat_df.head()

Clean and Allign

In [ ]:
stream = stream_df.copy()
snotel = snotel_df.copy()
landsat = landsat_df.copy()

if "Date" in stream.columns:
    stream["Date"] = pd.to_datetime(stream["Date"])
    stream = stream.set_index("Date")
else:
    stream.index = pd.to_datetime(stream.index)

if "Date" in snotel.columns:
    snotel["Date"] = pd.to_datetime(snotel["Date"])
    snotel = snotel.set_index("Date")
else:
    snotel.index = pd.to_datetime(snotel.index)

landsat.index = pd.to_datetime(landsat.index)

merged = pd.DataFrame(index=pd.date_range(StartDate, EndDate, freq="D"))

# streamflow
if "flow_cfs" in stream.columns:
    merged["streamflow_cfs"] = stream["flow_cfs"]
else:
    merged["streamflow_cfs"] = stream.iloc[:, 0]

# SNOTEL SWE
swe_col = "Snow Water Equivalent (m) Start of Day Values"
if swe_col in snotel.columns:
    merged["snotel_swe_m"] = snotel[swe_col]
else:
    merged["snotel_swe_m"] = snotel.iloc[:, 0]

# Landsat NDSI
merged["landsat_ndsi"] = landsat["NDSI"]

merged.head()

Plot

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

axes[0].plot(merged.index, merged["streamflow_cfs"])
axes[0].set_ylabel("cfs")
axes[0].set_title("USGS Streamflow")

axes[1].plot(merged.index, merged["snotel_swe_m"])
axes[1].set_ylabel("m")
axes[1].set_title("SNOTEL SWE")

axes[2].scatter(merged.index, merged["landsat_ndsi"], s=18)
axes[2].set_ylabel("NDSI")
axes[2].set_title("Landsat 8 NDSI")

axes[2].set_xlabel("Date")
plt.tight_layout()
plt.show()

Save

In [ ]:
merged.to_csv("Figures/watershed_stream_snotel_landsat.csv")
fig.savefig("Figures/watershed_stream_snotel_landsat.png", dpi=300, bbox_inches="tight")